# 03 · Don't Pay Twice

**Module 09 · Cost Optimization**

Real traffic repeats itself. "How much is express shipping?" arrives every day, in many spellings. Three kinds of caching, from automatic to clever:

| Kind | Who does it | What is saved | Risk |
|---|---|---|---|
| **Prompt caching** | OpenAI, automatically | about half the price of the repeated start of the prompt | none |
| **Exact cache** | us | the whole call, for the same question | none |
| **Semantic cache** | us | the whole call, for the same *meaning* | **it can give the wrong answer** |

In [1]:
import time

import cache
import inbox
import llm
import strategies

print("Ready")

Ready


## 1. Prompt caching: nothing to switch on
If the **start** of your prompt is identical to one OpenAI saw a moment ago, it does not process it again, and charges about half for those tokens. It is automatic, for prompts over 1,024 tokens. Our handbook is about 1,170 tokens, so it qualifies.

The only thing you control is the **order**: put the part that never changes **first** (the handbook) and the part that changes **last** (the customer's email).

To see it cleanly, we put a unique stamp at the very start, so the first call is guaranteed to be new:

In [2]:
stamp = "Session " + str(time.time()) + "\n\n"
questions = ["How much does express shipping cost?", "Can I pay with UPI?", "How long does standard shipping take?"]

print("GOOD ORDER: the handbook first, the email last")
for question in questions:
    answer = llm.ask(question, llm.SMALL, system_text=stamp + llm.HANDBOOK)
    print("  input tokens:", answer["prompt_tokens"], " cached:", str(answer["cached_tokens"]).rjust(4), "  cost: $" + format(answer["cost"], ".6f"))

GOOD ORDER: the handbook first, the email last


  input tokens: 1165  cached:    0   cost: $0.000186


  input tokens: 1165  cached: 1024   cost: $0.000113


  input tokens: 1165  cached: 1024   cost: $0.000112


The first call paid full price. The next ones found the handbook already processed. (OpenAI keeps it for a few minutes.)

Now the **bad order**: the changing part first. The start of every prompt is different, so there is nothing to reuse:

In [3]:
stamp = "Session " + str(time.time()) + "\n\n"

print("BAD ORDER: the email first, the handbook last")
for question in questions:
    # Everything in ONE message, with the question at the front. The shared handbook no longer starts the prompt.
    mixed = "Customer email: " + question + "\n\n" + llm.HANDBOOK
    answer = llm.ask(mixed, llm.SMALL, system_text=stamp + "You answer customer emails.")
    print("  input tokens:", answer["prompt_tokens"], " cached:", str(answer["cached_tokens"]).rjust(4), "  cost: $" + format(answer["cost"], ".6f"))

BAD ORDER: the email first, the handbook last


  input tokens: 1173  cached:    0   cost: $0.000194


  input tokens: 1173  cached:    0   cost: $0.000191


  input tokens: 1173  cached:    0   cost: $0.000201


Same words, same handbook, **no discount**, only because of the order. It is the cheapest optimisation there is: it costs nothing and cannot make an answer worse. **Rule: static first, changing last.**

## 2. The exact cache
Save the reply, keyed by the question. Before keying, we **normalise** the question, so capital letters, extra spaces and punctuation do not matter:

*From `cache.py`:*

```python
class AnswerCache:
    def __init__(self, use_semantic=True, threshold=SIMILARITY_THRESHOLD):
        self.use_semantic = use_semantic
        self.threshold = threshold
        self.exact = {}      # normalised question -> reply
        self.semantic = []   # a list of (vector, reply)
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0, "skipped": 0}

    def lookup(self, email):
        """Look for a saved answer. Returns a dictionary: reply (or None), kind, cost, vector."""
        if not is_cacheable(email):
            self.stats["skipped"] = self.stats["skipped"] + 1
            return {"reply": None, "kind": "skipped", "cost": 0.0, "vector": None}

        key = normalise(email)
        if key in self.exact:
            self.stats["exact_hits"] = self.stats["exact_hits"] + 1
            return {"reply": self.exact[key], "kind": "exact", "cost": 0.0, "vector": None}

        vector = None
        cost = 0.0
        if self.use_semantic and len(self.semantic) > 0:
            vectors, cost = llm.embed([email])
            vector = vectors[0]

            best_score = 0.0
            best_reply = None
            for saved_vector, saved_reply in self.semantic:
                score = llm.similarity(vector, saved_vector)
                if score > best_score:
                    best_score = score
                    best_reply = saved_reply

            if best_score >= self.threshold:
                self.stats["semantic_hits"] = self.stats["semantic_hits"] + 1
                return {"reply": best_reply, "kind": "semantic", "cost": cost, "vector": vector}

        self.stats["misses"] = self.stats["misses"] + 1
        return {"reply": None, "kind": "miss", "cost": cost, "vector": vector}

    def store(self, email, reply, vector=None):
        """Save an answer for next time. Only general questions are saved."""
        if not is_cacheable(email):
            return
        self.exact[normalise(email)] = reply

        if self.use_semantic:
            if vector is None:
                vectors, cost = llm.embed([email])
                vector = vectors[0]
            self.semantic.append((vector, reply))
```

In [4]:
for text in ["How much does express shipping cost?", "HOW MUCH does express shipping cost??  ", "how much does express shipping cost"]:
    print(repr(cache.normalise(text)))

'how much does express shipping cost'
'how much does express shipping cost'
'how much does express shipping cost'


All three become the same text, so the second and third questions would be free. An exact cache can never give a wrong answer for the *same* question.

## 3. The semantic cache: same meaning, different words
"What is the price of express delivery?" is not the same text, but it means the same. An **embedding** (Module 03) turns a question into numbers, so we can measure how close two questions are, from 0 to 1. If the closeness is above a **threshold**, we reuse the saved reply.

Let us look at the numbers. Some pairs **should** match, and some **must not**:

In [5]:
pairs = [
    ("SHOULD match", "How much does express shipping cost?", "What is the price of express delivery?"),
    ("SHOULD match", "Can I pay with UPI?", "Is UPI accepted as a payment method?"),
    ("SHOULD match", "How long is the warranty on a laptop?", "For how many months is a laptop covered by your warranty?"),
    ("must NOT", "How much does express shipping cost?", "How long does express shipping take?"),
    ("must NOT", "Can I pay with UPI?", "Can I pay with a credit card?"),
    ("must NOT", "Where is my order A1001?", "Where is my order A1002?"),
    ("must NOT", "My cart is $62. Will I pay anything for standard shipping?", "My cart is $30. Will I pay anything for standard shipping?"),
]

texts = []
for kind, first, second in pairs:
    texts.append(first)
    texts.append(second)
vectors, cost = llm.embed(texts)

print("Threshold we use:", cache.SIMILARITY_THRESHOLD)
print()
for number in range(len(pairs)):
    kind, first, second = pairs[number]
    score = llm.similarity(vectors[2 * number], vectors[2 * number + 1])
    verdict = "would reuse the saved reply"
    if score < cache.SIMILARITY_THRESHOLD:
        verdict = "would call the model"
    print(format(score, ".3f"), "|", kind.ljust(12), "|", verdict.ljust(28), "|", first[:48], "<->", second[:40])

Threshold we use: 0.8

0.808 | SHOULD match | would reuse the saved reply  | How much does express shipping cost? <-> What is the price of express delivery?
0.834 | SHOULD match | would reuse the saved reply  | Can I pay with UPI? <-> Is UPI accepted as a payment method?
0.832 | SHOULD match | would reuse the saved reply  | How long is the warranty on a laptop? <-> For how many months is a laptop covered 
0.766 | must NOT     | would call the model         | How much does express shipping cost? <-> How long does express shipping take?
0.593 | must NOT     | would call the model         | Can I pay with UPI? <-> Can I pay with a credit card?
0.958 | must NOT     | would reuse the saved reply  | Where is my order A1001? <-> Where is my order A1002?
0.903 | must NOT     | would reuse the saved reply  | My cart is $62. Will I pay anything for standard <-> My cart is $30. Will I pay anything for 


Look at the three "must NOT" pairs near the bottom:

- **"express cost" and "express time"** score high even though they ask different things. A threshold is a trade-off: lower it and you catch more paraphrases, but you start returning **wrong answers**.
- **Order A1001 and order A1002** score above 0.95. To an embedding they are nearly the same question. For the customer they are two different orders. Returning the other person's order status would be a **data leak**.
- **A $62 cart and a $30 cart** also match, but one pays shipping and the other does not.

So the semantic cache needs **guard rails**: only cache **general** questions. Anything with a number or a personal detail is never cached:

*From `cache.py`:*

```python
def is_cacheable(email):
    """True only for general questions. An email with a number or a personal detail needs its own answer."""
    if re.search(r"\d", email):          # any digit: an order id, an amount, a date...
        return False

    text = email.lower()
    for word in PERSONAL_WORDS:
        if word in text:
            return False
    return True
```

In [6]:
for email in ["Do you ship to Canada?", "Where is my order A1001?", "My cart is $62. Will I pay anything for standard shipping?", "Can I pay with UPI?"]:
    print(str(cache.is_cacheable(email)).ljust(6), "|", email)

True   | Do you ship to Canada?
False  | Where is my order A1001?
False  | My cart is $62. Will I pay anything for standard shipping?
True   | Can I pay with UPI?


## 4. A day of traffic
The test set has 28 different emails. A real day also has **repeats**: 4 exact repeats (different capitals and spacing) and 4 paraphrases, 36 emails in all. We run the day on the **strong** model, without and with the cache:

In [7]:
day = inbox.traffic()
print("Emails in the day:", len(day))

without = strategies.evaluate(strategies.always_big, day)

answer_cache = cache.AnswerCache()
with_cache = strategies.evaluate(strategies.WithCache(strategies.always_big, answer_cache), day)

for name, row in [("without cache", without), ("with cache", with_cache)]:
    print(name.ljust(14), "$ per 1,000: " + format(row["cost_per_1000"], ".2f").rjust(6),
          "  all facts: " + str(row["perfect"]) + "/" + str(row["emails"]),
          "  avg seconds: " + format(row["avg_seconds"], ".2f"))

print()
print("What the cache did:", answer_cache.stats)
print("Where the replies came from:", with_cache["sources"])
for miss in with_cache["misses"]:
    print("MISSED A FACT:", miss["email"][:70], "->", miss["source"])

Emails in the day: 36


without cache  $ per 1,000:   1.91   all facts: 36/36   avg seconds: 0.72
with cache     $ per 1,000:   1.54   all facts: 36/36   avg seconds: 0.72

What the cache did: {'exact_hits': 4, 'semantic_hits': 3, 'misses': 13, 'skipped': 16}
Where the replies came from: {'gpt-4o': 29, 'exact cache': 4, 'semantic cache': 3}


Read the cache statistics:

- **exact hits** and **semantic hits**: calls we did not pay for.
- **skipped**: emails the guard refused to cache (they contain a number or a personal detail). Most of a real desk's emails are personal, which is why a cache saves **less than people hope**.
- **misses**: general questions seen for the first time. They were answered and saved for next time.

## 5. Does caching always pay? Do the arithmetic.
A cache hit saves the whole call. How much is that worth with each model?

In [8]:
typical_input = 1170     # tokens: the handbook plus the question
typical_output = 40      # tokens: a short reply

for model in [llm.SMALL, llm.BIG]:
    one_call = llm.cost_of(model, typical_input, 0, typical_output)
    print(model.ljust(12), "one call: $" + format(one_call, ".6f"), "   so 1,000 cache hits save $" + format(one_call * 1000, ".2f"))

gpt-4o-mini  one call: $0.000199    so 1,000 cache hits save $0.20
gpt-4o       one call: $0.003325    so 1,000 cache hits save $3.32


With the cheap model, **a cache hit saves about two hundredths of a cent** (1,000 hits save about 20 cents). The lookup itself is a far smaller cost than that, so it does pay, but only at high volume. With the strong model, a hit is worth about 17 times more.

> **Lesson: a cache is worth building when what it saves is large compared with what it costs to run.** For a cheap model the money saved is small, and the saving is mostly **speed** (no model call). A cache is also more code that can be wrong. Do the arithmetic before adding it.

## What you learned
- **Prompt caching** is free and automatic. Your only job: static first, changing last.
- An **exact cache** is safe. A **semantic cache** can be wrong, so it needs a threshold you have measured and a guard that keeps personal emails out.
- Caches save less than expected on a desk full of personal emails, and little on a cheap model.

**Next:** send less, and wait for a discount.